In [1]:
import pandas as pd, numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import TargetEncoder
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

df = pd.read_parquet("../out/ksa_sales_clean.parquet")
m = df[(df.classification == "سكني") & (~df.is_multi_property)
       & (df.area_m2.between(50, 5_000))
       & (df.price_sar.between(50_000, 20_000_000))].copy()

# features
m["log_area"] = np.log(m.area_m2)
m["t"] = (m.date_greg - pd.Timestamp("2020-01-01")).dt.days / 365.25
y = np.log1p(m.price_sar)

cut = pd.Timestamp("2025-01-01")
itr, ite = m.date_greg < cut, m.date_greg >= cut
X = m[["log_area", "t", "location", "city", "region"]]

pre = ColumnTransformer([
    ("num", "passthrough", ["log_area", "t"]),
    ("cat", TargetEncoder(target_type="continuous", random_state=0),
            ["location", "city", "region"]),
])

pipe = Pipeline([
    ("pre", pre),
    ("model", HistGradientBoostingRegressor(
        max_iter=400, learning_rate=0.1, min_samples_leaf=50, random_state=0)),
])

pipe.fit(X[itr], y[itr])
p = pipe.predict(X[ite])

pred, actual = np.expm1(p), m.price_sar[ite]
print("MODEL, test = 2025+")
print("  R²  (log):", round(r2_score(y[ite], p), 4))
print("  MAE (log):", round(mean_absolute_error(y[ite], p), 4))
print("  median abs % error:", f"{np.median(np.abs(pred-actual)/actual):.1%}")
print("  bias:", round(float(np.median(pred/actual)), 3))

MODEL, test = 2025+
  R²  (log): 0.661
  MAE (log): 0.4124
  median abs % error: 27.8%
  bias: 0.989


In [2]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

def run(name, model, scale):
    pre = ColumnTransformer([
        ("num", StandardScaler() if scale else "passthrough", ["log_area", "t"]),
        ("cat", TargetEncoder(target_type="continuous"), ["location", "city", "region"]),
    ])
    pipe = Pipeline([("pre", pre), ("model", model)])
    pipe.fit(X[itr], y[itr])
    p = pipe.predict(X[ite])
    pred, actual = np.expm1(p), m.price_sar[ite]
    print(f"{name:22s} R² {r2_score(y[ite], p):.4f} | MAE {mean_absolute_error(y[ite], p):.4f} "
          f"| med%err {np.median(np.abs(pred-actual)/actual):.1%} "
          f"| bias {np.median(pred/actual):.3f}")

run("Ridge (linear)", Ridge(alpha=1.0), True)
run("HistGradientBoosting", HistGradientBoostingRegressor(
        max_iter=400, learning_rate=0.1, min_samples_leaf=50, random_state=0), False)

Ridge (linear)         R² 0.5953 | MAE 0.4688 | med%err 34.5% | bias 1.076
HistGradientBoosting   R² 0.6618 | MAE 0.4118 | med%err 27.6% | bias 0.988


In [3]:
from sklearn.model_selection import cross_val_score, KFold
from sklearn.tree import DecisionTreeRegressor

def make_pipe(num_cols=("log_area", "t")):
    return Pipeline([
        ("pre", ColumnTransformer([
            ("num", "passthrough", list(num_cols)),
            ("cat", TargetEncoder(target_type="continuous"),
                    ["location", "city", "region"]),
        ])),
        ("model", HistGradientBoostingRegressor(
            max_iter=400, learning_rate=0.1, min_samples_leaf=50, random_state=0)),
    ])

# --- 1. the dishonest split, for comparison -------------------------
scores = cross_val_score(make_pipe(), X, y,
                         cv=KFold(5, shuffle=True, random_state=0), scoring="r2")
print("shuffled KFold R²:", scores.round(4), "mean", round(scores.mean(), 4))
print("time-ordered   R²: 0.661")
print("gap            :", round(scores.mean() - 0.661, 4))

shuffled KFold R²: [0.7277 0.7289 0.7275 0.7261 0.7264] mean 0.7273
time-ordered   R²: 0.661
gap            : 0.0663


In [4]:
# --- 2. single-feature leak scan ------------------------------------
sub = m[itr].sample(120_000, random_state=0)
ysub = np.log1p(sub.price_sar)

for c in ["log_area", "t", "area_m2"]:
    s = cross_val_score(DecisionTreeRegressor(max_depth=3, random_state=0),
                        sub[[c]], ysub, cv=3, scoring="r2").mean()
    print(f"  {c:12s} {s:.3f}")

for c in ["location", "city", "region"]:
    enc = TargetEncoder(target_type="continuous").fit_transform(sub[[c]], ysub)
    s = cross_val_score(DecisionTreeRegressor(max_depth=3, random_state=0),
                        enc, ysub, cv=3, scoring="r2").mean()
    print(f"  {c:12s} {s:.3f}  (target-encoded)")

  log_area     0.059
  t            0.036
  area_m2      0.059
  location     0.560  (target-encoded)
  city         0.382  (target-encoded)
  region       0.141  (target-encoded)


In [5]:
# --- 3. what the leak would have done -------------------------------
m["price_per_m2"] = m.price_sar / m.area_m2     # the column we dropped in cleaning
X_leak = m[["log_area", "t", "price_per_m2", "location", "city", "region"]]

p_leak = make_pipe(["log_area", "t", "price_per_m2"]).fit(
    X_leak[itr], y[itr]).predict(X_leak[ite])
print("clean model       R²:", round(r2_score(y[ite], p1), 4))
print("with price_per_m2 R²:", round(r2_score(y[ite], p_leak), 4))

NameError: name 'p1' is not defined